<a href="https://colab.research.google.com/github/VasilisPapageorgiou/Statistical-Inference-Finite-Epidemics/blob/main/Sim2_SI_for_Epidemic_Risk_Functionals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.stats import chi2

# ============================================================
# SIMULATION 2 — DESCRIPTOR-LEVEL UNCERTAINTY GEOMETRY
# Exact two-regime CTMC; complete-data maximum likelihood
# Analytical recursive risk sensitivities (Proposition 3)
# First-order risk envelopes based on Theorem 5
#
# ONE RESULTS TABLE — NO FIGURES — NO GOOGLE DRIVE
# Run everything in ONE Google Colab cell.
# ============================================================

CFG = dict(
    N=50,
    i0=2,
    gamma=1.0,
    m=400,
    MC=500,
    alpha=0.01,
    seed=20261009,
    thresholds=(5, 12, 24, 36, 43),
    cases={
        "Moderate": (
            (1.45, 0.75, 0.35, 0.25), "stationary", 0
        ),
        "Weak / relevant": (
            (1.65, 0.70, 0.04, 0.08), "phase2", 0
        ),
        "Weak / near-orthogonal": (
            (1.45, 0.70, 0.02, 0.08), "phase1", 0
        ),
    },
    nonlinear_every=0
)

rng = np.random.default_rng(CFG["seed"])

N = CFG["N"]
i0 = CFG["i0"]
s0 = N - i0
gamma = CFG["gamma"]
C = np.asarray(CFG["thresholds"], dtype=int)

assert np.all((C >= 0) & (C < s0))
assert len(np.unique(C)) == len(C)

# ============================================================
# 1. SIMULATE ONE COMPLETE CTMC TRAJECTORY
#
# Unknown: b1, b2, q12, q21
# Known: gamma
#
# Sufficient statistics:
# Event counts for infections and regime changes
# Integrated exposures for infection and switching rates
# ============================================================

def simulate(theta, start_mode):

    b1, b2, q12, q21 = theta

    b = np.array([b1, b2])
    q = np.array([q12, q21])

    if start_mode == "stationary":
        p = int(
            rng.random() >= q21 / (q12 + q21)
        )
    elif start_mode == "phase1":
        p = 0
    else:
        p = 1

    s, i = s0, i0

    cnt = np.zeros(4)
    expo = np.zeros(4)

    while i > 0:

        infection = b[p] * s * i / N
        recovery = gamma * i
        switching = q[p]

        total = infection + recovery + switching

        dt = rng.exponential(1 / total)

        expo[p] += s * i / N * dt
        expo[2 + p] += dt

        u = rng.random() * total

        if u < infection:
            cnt[p] += 1
            s -= 1
            i += 1

        elif u < infection + recovery:
            i -= 1

        else:
            cnt[2 + p] += 1
            p = 1 - p

    return cnt, expo


# ============================================================
# 2. EXACT FINAL-SIZE RECURSION AND ANALYTICAL JACOBIAN
#
# Parameters are represented on the log scale:
# eta = (log b1, log b2, log q12, log q21).
#
# Differentiation:
# A x = rhs
# A dx = d(rhs) - (dA)x
# ============================================================

def risks_and_jac(eta):

    rates = np.exp(eta)
    b = rates[:2]
    q = rates[2:]

    X = {}
    D = {}

    for s in range(s0 + 1):

        for i in range(N - s + 1):

            x = np.zeros((s + 1, 2))
            dx = np.zeros((s + 1, 2, 4))

            if i == 0 or s == 0:

                x[0] = 1.0

                X[s, i] = x
                D[s, i] = dx
                continue

            infection = b * s * i / N

            A = np.array([
                [
                    infection[0] + gamma * i + q[0],
                    -q[0]
                ],
                [
                    -q[1],
                    infection[1] + gamma * i + q[1]
                ]
            ])

            dA = np.zeros((4, 2, 2))

            for p in range(2):

                dA[p, p, p] = infection[p]

                dA[2 + p, p, p] += q[p]

                dA[2 + p, p, 1 - p] -= q[p]

            for n in range(s + 1):

                rhs = (
                    gamma * i
                    * X[s, i - 1][n].copy()
                )

                drhs = (
                    gamma * i
                    * D[s, i - 1][n].copy()
                )

                if n > 0:

                    lower = X[s - 1, i + 1][n - 1]

                    rhs += infection * lower

                    drhs += (
                        infection[:, None]
                        * D[s - 1, i + 1][n - 1]
                    )

                    for p in range(2):

                        drhs[p, p] += (
                            infection[p] * lower[p]
                        )

                x[n] = np.linalg.solve(A, rhs)

                for k in range(4):

                    dx[n, :, k] = np.linalg.solve(
                        A,
                        drhs[:, k] - dA[k] @ x[n]
                    )

            X[s, i] = x
            D[s, i] = dx

    pmf = X[s0, i0][:, 0]
    dp = D[s0, i0][:, 0, :]

    if abs(pmf.sum() - 1) > 1e-7:
        raise ArithmeticError(
            "Final-size probabilities do not sum to one."
        )

    risks = np.array([
        pmf[c + 1:].sum()
        for c in C
    ])

    jac = np.array([
        dp[c + 1:].sum(axis=0)
        for c in C
    ])

    return risks, jac


# ============================================================
# 3. MONTE CARLO EXPERIMENT FOR ONE IDENTIFICATION REGIME
# ============================================================

def compute_case(true_rates, start_mode, target_phase):

    assert target_phase == 0

    eta0 = np.log(true_rates)

    truth, _ = risks_and_jac(eta0)

    critical = np.sqrt(
        chi2.ppf(1 - CFG["alpha"], 4)
    )

    results = []
    failures = 0

    for replicate in range(CFG["MC"]):

        cnt = np.zeros(4)
        exposure = np.zeros(4)

        # Generate m independent complete outbreaks

        for _ in range(CFG["m"]):

            a, e = simulate(
                true_rates, start_mode
            )

            cnt += a
            exposure += e

        # Complete-data MLE.
        # All four estimated event counts must be positive.

        if (
            np.any(cnt == 0)
            or np.any(exposure == 0)
        ):
            failures += 1
            continue

        fitted = cnt / exposure

        eta = np.log(fitted)

        # Observed information covariance on log scale.
        # Cov(eta_hat) approximately diag(1/counts).

        cov = np.diag(1 / cnt)

        try:
            est, J = risks_and_jac(eta)

        except (
            FloatingPointError,
            ArithmeticError,
            np.linalg.LinAlgError
        ):
            failures += 1
            continue

        # Theorem 5: projected parameter covariance

        V = np.einsum(
            "ci,ij,cj->c",
            J, cov, J
        )

        half_width = (
            critical
            * np.sqrt(np.maximum(V, 0))
        )

        # First-order descriptor envelopes.
        # Clipping respects risk probability bounds.

        lower = np.maximum(
            0, est - half_width
        )

        upper = np.minimum(
            1, est + half_width
        )

        covered = np.all(
            (truth >= lower - 1e-12)
            & (truth <= upper + 1e-12)
        )

        # Leading uncertainty direction

        eigenvalues, eigenvectors = np.linalg.eigh(cov)

        leading_direction = eigenvectors[:, -1]

        # Absolute gradient alignment

        alignment = (
            np.abs(J @ leading_direction)
            / (np.linalg.norm(J, axis=1) + 1e-14)
        )

        tail = C >= s0 // 2

        row = dict(
            coverage=float(covered),

            width=np.mean(upper - lower),

            tail_width=np.mean(
                (upper - lower)[tail]
            ),

            trace_cov=np.trace(cov),

            lambda_max=eigenvalues[-1],

            alignment=np.mean(alignment),

            tail_alignment=np.mean(
                alignment[tail]
            )
        )

        results.append(row)

        if (replicate + 1) % 100 == 0:

            print(
                f"  Completed {replicate + 1}"
                f"/{CFG['MC']} replications",
                flush=True
            )

    return results, failures


# ============================================================
# 4. RUN ALL THREE IDENTIFICATION REGIMES
# ============================================================

records = []
all_results = []

print("=" * 65)
print("SIMULATION 2 — DESCRIPTOR UNCERTAINTY GEOMETRY")
print("=" * 65)

print("Nominal confidence level:", 1 - CFG["alpha"])
print("Population size:", N)
print("Independent outbreaks per replication:", CFG["m"])
print("Monte Carlo replications:", CFG["MC"])
print("Thresholds:", C)

for name, (rates, initial_mode, target_phase) in (
    CFG["cases"].items()
):

    print(f"\n{name}: starting", flush=True)

    results, failures = compute_case(
        np.asarray(rates, dtype=float),
        initial_mode,
        target_phase
    )

    if not results:
        raise RuntimeError(
            f"No valid replications for {name}."
        )

    df = pd.DataFrame(results)

    coverage = float(df["coverage"].mean())

    mcse = np.sqrt(
        coverage * (1 - coverage) / len(df)
    )

    records.append(dict(
        Case=name,
        MC=CFG["MC"],
        Valid=len(df),
        Failed=failures,

        Full_coverage=round(coverage, 3),
        Coverage_MCSE=round(mcse, 3),

        Mean_width=round(
            df["width"].mean(), 3
        ),

        Tail_width=round(
            df["tail_width"].mean(), 3
        ),

        Trace_cov=round(
            df["trace_cov"].mean(), 4
        ),

        Largest_eigenvalue=round(
            df["lambda_max"].mean(), 4
        ),

        Mean_alignment=round(
            df["alignment"].mean(), 3
        ),

        Tail_alignment=round(
            df["tail_alignment"].mean(), 3
        )
    ))

    all_results.append(
        df.assign(Case=name)
    )


# ============================================================
# 5. FINAL TABLE — NO FIGURES
# ============================================================

summary = pd.DataFrame(records)

replications = pd.concat(
    all_results,
    ignore_index=True
)

summary.to_csv(
    "simulation2_summary.csv",
    index=False
)

replications.to_csv(
    "simulation2_replications.csv",
    index=False
)

print("\n" + "=" * 65)
print("FINAL RESULTS — SIMULATION 2")
print("=" * 65)

print(summary.to_string(index=False))

print("\nSaved files:")
print("simulation2_summary.csv")
print("simulation2_replications.csv")

print(
    "\nCoverage is conditional on successful estimation."
)

print(
    "Envelopes use the first-order approximation from Theorem 5."
)

print(
    "Parameter covariance and gradients are calculated on the log scale."
)

print("\nSIMULATION 2 COMPLETED.")


SIMULATION 2 — DESCRIPTOR UNCERTAINTY GEOMETRY
Nominal confidence level: 0.99
Population size: 50
Independent outbreaks per replication: 100
Monte Carlo replications: 500
Thresholds: [ 5 12 24 36 43]

Moderate: starting
  Completed 100/500 replications
  Completed 200/500 replications
  Completed 300/500 replications
  Completed 400/500 replications
  Completed 500/500 replications

Weak / relevant: starting
  Completed 100/500 replications
  Completed 200/500 replications
  Completed 300/500 replications
  Completed 400/500 replications
  Completed 500/500 replications

Weak / near-orthogonal: starting
  Completed 200/500 replications
  Completed 300/500 replications
  Completed 400/500 replications
  Completed 500/500 replications

FINAL RESULTS — SIMULATION 2
                  Case  MC  Valid  Failed  Full_coverage  Coverage_MCSE  Mean_width  Tail_width  Trace_cov  Largest_eigenvalue  Mean_alignment  Tail_alignment
              Moderate 500    500       0          0.950          0.